In [205]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,Id,Versuri,Autor
0,Zho5dMCjumAcYUawccJqMy,"Pe barbari de-i risipeşte,\nŞ-apoi vecinic pri...",Vasile Alecsandri
1,DphNgshYSbs9vZGUempJMG,Sau va avea mulţi copii.\nPentru că totuna est...,Grigore Vieru
2,KnZnz7NwJqW7zdgnUFwk6R,"Copilăriei noastre.\nFrate, ei urăsc\nCântecul...",Grigore Vieru
3,YWnVtA7h2icwHWFzbuAjSu,"Veniţi lângă mine, tovarăşi! Că mâne-o să mor,...",Lucian Blaga
4,cSieLQMAQLGD5UutQ8hw7k,"Iar tu, Hyperion, rămâi\nOriunde ai apune...\n...",Mihai Eminescu


In [206]:
df_train.Versuri[0]

'Pe barbari de-i risipeşte,\nŞ-apoi vecinic priveghează,\nSentinelă mult vetează,\nŞi te-aţine la hotare'

In [207]:
import re
#niciun preprocessing pentru a pastra stilul autorului
def preprocess(text):
    text = str(text)
    return text

preprocess(df_train.Versuri[0])

'Pe barbari de-i risipeşte,\nŞ-apoi vecinic priveghează,\nSentinelă mult vetează,\nŞi te-aţine la hotare'

In [208]:
df_train.Versuri = df_train.Versuri.apply(preprocess)
df_test.Versuri = df_test.Versuri.apply(preprocess)

df_train.head()

,Id,Versuri,Autor
0,Zho5dMCjumAcYUawccJqMy,"Pe barbari de-i risipeşte,\nŞ-apoi vecinic pri...",Vasile Alecsandri
1,DphNgshYSbs9vZGUempJMG,Sau va avea mulţi copii.\nPentru că totuna est...,Grigore Vieru
2,KnZnz7NwJqW7zdgnUFwk6R,"Copilăriei noastre.\nFrate, ei urăsc\nCântecul...",Grigore Vieru
3,YWnVtA7h2icwHWFzbuAjSu,"Veniţi lângă mine, tovarăşi! Că mâne-o să mor,...",Lucian Blaga
4,cSieLQMAQLGD5UutQ8hw7k,"Iar tu, Hyperion, rămâi\nOriunde ai apune...\n...",Mihai Eminescu


In [209]:
#creez feature uri numerice noi

def lungime_medie_vers(text):
    versuri = [v.strip() for v in text.split('\n')]
    return sum(len(v) for v in versuri) / len(versuri)

df_train['Nr caractere'] = df_train.Versuri.map(lambda x : len(x))
df_train['Nr cuvinte'] = df_train.Versuri.map(lambda x : len(x.split(" ")))
df_train['Lungime medie vers'] = df_train.Versuri.map(lungime_medie_vers)

df_train['Nr virgule'] = df_train.Versuri.map(lambda x : x.count(','))
df_train['Nr puncte'] = df_train.Versuri.map(lambda x : x.count('.'))

df_test['Nr caractere'] = df_test.Versuri.map(lambda x : len(x))
df_test['Nr cuvinte'] = df_test.Versuri.map(lambda x : len(x.split(" ")))
df_test['Lungime medie vers'] = df_test.Versuri.map(lungime_medie_vers)

df_test['Nr virgule'] = df_test.Versuri.map(lambda x : x.count(','))
df_test['Nr puncte'] = df_test.Versuri.map(lambda x : x.count('.'))

df_train.head()

,Id,Versuri,Autor,Nr caractere,Nr cuvinte,Lungime medie vers,Nr virgule,Nr puncte
0,Zho5dMCjumAcYUawccJqMy,"Pe barbari de-i risipeşte,\nŞ-apoi vecinic pri...",Vasile Alecsandri,100,11,24.25,3,0
1,DphNgshYSbs9vZGUempJMG,Sau va avea mulţi copii.\nPentru că totuna est...,Grigore Vieru,96,15,23.25,1,2
2,KnZnz7NwJqW7zdgnUFwk6R,"Copilăriei noastre.\nFrate, ei urăsc\nCântecul...",Grigore Vieru,67,7,16.00,1,2
3,YWnVtA7h2icwHWFzbuAjSu,"Veniţi lângă mine, tovarăşi! Că mâne-o să mor,...",Lucian Blaga,106,17,25.75,3,0
4,cSieLQMAQLGD5UutQ8hw7k,"Iar tu, Hyperion, rămâi\nOriunde ai apune...\n...",Mihai Eminescu,97,12,23.50,2,3


In [210]:
from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
df_train.Autor = enc.fit_transform(df_train.Autor)

df_train.head()

,Id,Versuri,Autor,Nr caractere,Nr cuvinte,Lungime medie vers,Nr virgule,Nr puncte
0,Zho5dMCjumAcYUawccJqMy,"Pe barbari de-i risipeşte,\nŞ-apoi vecinic pri...",6,100,11,24.25,3,0
1,DphNgshYSbs9vZGUempJMG,Sau va avea mulţi copii.\nPentru că totuna est...,3,96,15,23.25,1,2
2,KnZnz7NwJqW7zdgnUFwk6R,"Copilăriei noastre.\nFrate, ei urăsc\nCântecul...",3,67,7,16.00,1,2
3,YWnVtA7h2icwHWFzbuAjSu,"Veniţi lângă mine, tovarăşi! Că mâne-o să mor,...",4,106,17,25.75,3,0
4,cSieLQMAQLGD5UutQ8hw7k,"Iar tu, Hyperion, rămâi\nOriunde ai apune...\n...",5,97,12,23.50,2,3


In [211]:
features = 'Versuri'
target = 'Autor'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer(analyzer='char', ngram_range=((3, 5))) #analyzer pe charactere tot asa pentru a pastra in continuare stilul autorului
X = vect.fit_transform(X)
X_final = vect.transform(X_final)

X

<3415x143848 sparse matrix of type '<class 'numpy.float64'>'
	with 1073150 stored elements in Compressed Sparse Row format>

In [212]:
df_test.columns

Index(['Id', 'Versuri', 'Nr caractere', 'Nr cuvinte', 'Lungime medie vers',
       'Nr virgule', 'Nr puncte'],
      dtype='object')

In [213]:
other_features = ['Nr caractere', 'Nr cuvinte', 'Lungime medie vers', 'Nr virgule', 'Nr puncte']

X_numeric = df_train[other_features]
X_numeric_final = df_test[other_features]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_numeric = pd.DataFrame(scaler.fit_transform(X_numeric), columns=other_features)
X_numeric_final = pd.DataFrame(scaler.transform(X_numeric_final), columns=other_features)

X_numeric_final.head()

,Nr caractere,Nr cuvinte,Lungime medie vers,Nr virgule,Nr puncte
0,-0.534925,-0.561311,-0.534925,-1.294638,-0.863645
1,1.129242,1.174073,1.129242,-1.294638,-0.202356
2,-1.123784,-0.705926,-1.123784,-0.699832,-0.202356
3,-0.074079,-0.127465,-0.074079,-0.105027,0.458933
4,1.794909,1.318688,1.794909,-1.294638,4.426667


In [214]:
from scipy.sparse import hstack

X = hstack([X, X_numeric])
X_final = hstack([X_final, X_numeric_final])

X

<3415x143853 sparse matrix of type '<class 'numpy.float64'>'
	with 1090225 stored elements in COOrdinate format>

In [215]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

model = LinearSVC(C=7.0, max_iter=10000)

cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()

0.7320644216691068

In [216]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [217]:
y_preds = enc.inverse_transform(y_preds)
y_preds[:5]

array(['Grigore Vieru', 'Grigore Vieru', 'Grigore Vieru', 'Lucian Blaga',
       'Mihai Eminescu'], dtype=object)

In [218]:
submission = pd.DataFrame({
    "Id": df_test.Id,
    "Autor": y_preds
})

submission

,Id,Autor
0,7oYYGUufTgpKPCcVUhTpAS,Grigore Vieru
1,EGWuQxkFSjW59anvGxW7nS,Grigore Vieru
2,XfHVBFedR9cDb4DBvuwtsF,Grigore Vieru
3,E3EbfZuVsZhUxqRYvBbywM,Lucian Blaga
4,kAxEyKrjhpvUu25CAjh5F6,Mihai Eminescu
...,...,...
849,78iMMRHBxQAmtn3tjDronb,Ana Blandiana
850,9u7WHXTNKeppwQbVHfBaEq,Grigore Vieru
851,GuP57mZ2opQC2AMrdV7u4q,Grigore Vieru
852,4GN8jWtF5KUDfH49oGmXNg,Grigore Vieru


In [219]:
submission.to_csv("submission.csv", index=False)